# Fault injection run

One run = one model, one format, one task, one compute dtype. Pick them in the config cell, then
*Runtime > Run all*. Use an **L4** runtime. If the session dies, run all cells again; finished
trials are skipped.

Results are written to `RESULTS_ROOT` on Drive. The team keeps one shared folder
`LLM-ScaleBit-Results` (added to everyone's My Drive as a shortcut).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO = "https://github.com/mehmetd7mir/LLM-ScaleBit-SoftError.git"
CODE = "/content/LLM-ScaleBit-SoftError"
COMMIT = None   # set a commit hash to rerun with an exact code version

import os, subprocess, sys
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
if COMMIT:
    subprocess.run(["git", "-C", CODE, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)
sys.path.insert(0, CODE)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
# Hugging Face token: needed for gated models (Llama). Add it once under the key icon
# (Colab Secrets) with the name HF_TOKEN.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("no HF_TOKEN secret (fine for Qwen models)")

## Config

| field | values |
|---|---|
| `fmt` | `fp`, `int4_rtn`, `int4_gptq`, `mxfp4`, `nvfp4` |
| `task` | `mmlu`, `gsm8k` |
| `dtype` | `float16` (main runs), `bfloat16` (dtype sub-study) |
| `gptq_id` | only for `int4_gptq`, e.g. `Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4` |
| `trials_per_bit` | trials per bit position (300 in the main runs) |
| `trials_override` | per-class count, e.g. `{"element": 1000, "tensor_scale": 50}` |

GSM8K generates up to 512 tokens; goldens that hit the limit are left out of the trials.

In [ ]:
from scalebit.run import Config, Experiment

RESULTS_ROOT = "/content/drive/MyDrive/LLM-ScaleBit-Results"

cfg = Config(
    model_id="Qwen/Qwen2.5-7B-Instruct",
    fmt="mxfp4",
    task="mmlu",
    dtype="float16",
    gptq_id=None,
    trials_per_bit=300,
    trials_override={"element": 1000, "tensor_scale": 30},
    study="main",
    results_root=RESULTS_ROOT,
)
cfg

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > L4"
gpu = torch.cuda.get_device_name(0)
print(gpu, f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")
if "L4" not in gpu:
    print("WARNING: the study runs on L4. Do not mix GPU types inside one study.")
# ECC state of this GPU (the fault model does not depend on it; we report it in the paper)
try:
    ecc = subprocess.run(["nvidia-smi", "--query-gpu=ecc.mode.current", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    print("ECC:", ecc or "unknown")
except FileNotFoundError:
    print("ECC: nvidia-smi not found")

## Load, convert, golden run, checks
All three checks must pass before any trial runs.

In [ ]:
ex = Experiment(cfg).setup()
ex.golden()
ok_det = ex.check_determinism()
ok_san = ex.check_sanity()
print(f"memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GB")
assert ok_det and ok_san, "A check failed. Stop here and report the output above."

## Trials

In [ ]:
ex.run()

## Persistent fault (optional)

A memory fault stays until the model is reloaded. This flips one bit, runs every question with
the fault in place, restores, and repeats for 20 random fields. Set `PERSIST` to a
`(class, bit)` pair, e.g. `("scale", 7)` for MXFP4, `("scale", 14)` for INT4.

In [ ]:
PERSIST = None   # e.g. ("scale", 7)
if PERSIST:
    ex.run_persistent(*PERSIST, n_faults=20)